In [ ]:
# !pip install tensorflow

In [ ]:
# !pip install xgboost

In [ ]:
# !pip install numpy==1.24.3

In [ ]:
# Import Libraries and Dependencies
import pandas as pd
import numpy as np
import re
# import spacy
import subprocess
import os
import matplotlib.pyplot as plt
import seaborn as sns  
from sklearn.model_selection import train_test_split    # To split the dataset into train and test sets
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder   # For scaling and encoding categorical variables
from sklearn.compose import ColumnTransformer           # To apply transformers to specific columns
from sklearn.impute import SimpleImputer                # For handling missing data
from sklearn.ensemble import RandomForestClassifier     # Random Forest Classifier
from sklearn.linear_model import LogisticRegression     # Logistic Regression
from sklearn.svm import SVC                            # Support Vector Classifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score # For evaluating the model
import joblib
from xgboost import XGBClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV


import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Concatenate
from tensorflow.keras.models import Model


In [ ]:
!pip show tensorflow

In [ ]:
# Read in the CSV file
disease_df = pd.read_csv('Resources/disease_symptoms.csv')

In [ ]:
# Rename Disease column to Prognosis
disease_df.rename(columns={'Disease': 'Prognosis'}, inplace=True)

In [ ]:

# Expand Symptoms into separate columns
disease_df_expanded = disease_df["Symptoms"].str.split(", ", expand=True)

In [ ]:

# Rename columns dynamically
disease_df_expanded.columns = [f"Symptom_{i+1}" for i in range(disease_df_expanded.shape[1])]

In [ ]:

# Concatenate with Prognosis column
disease_df_final = pd.concat([disease_df["Prognosis"], disease_df_expanded], axis=1)

In [ ]:

# Reshape from wide to long format
disease_df_long = disease_df_final.melt(id_vars="Prognosis", value_name="Symptom").drop(columns=["variable"])

In [ ]:
# Drop NaN values before encoding**
disease_df_long = disease_df_long.dropna(subset=["Symptom"])

In [ ]:
# **Fix: Trim whitespace from symptoms**
disease_df_long["Symptom"] = disease_df_long["Symptom"].str.strip()

In [ ]:
# One-hot encode symptoms properly
disease_df_encoded = pd.get_dummies(disease_df_long, columns=["Symptom"])

In [ ]:
# **Fix: Ensure index is reset**
disease_df_encoded = disease_df_encoded.reset_index(drop=True)

In [ ]:
# sort alphabetically by Prognosis
disease_df_encoded = disease_df_encoded.sort_values("Prognosis")

In [ ]:
# disease_df_encoded.set_index('Prognosis', inplace=True)


In [ ]:
#reset the index 
disease_df_encoded = disease_df_encoded.reset_index(drop=True)
#Display the result
print(disease_df_encoded.shape)
disease_df_encoded.head(50)

In [ ]:
# remove Symptom_ in the column names
disease_df_encoded.columns = disease_df_encoded.columns.str.replace('Symptom_', '')

# Display the result
disease_df_encoded.head(50)

In [ ]:
disease_df_encoded.shape

In [ ]:
# read in prognosis csv file
prognosis_df = pd.read_csv('Resources/symbipredict_2022.csv')

# Display the result
prognosis_df.head()

In [ ]:
# Prognosis column to first column in the dataset
prognosis_df = prognosis_df[["prognosis"] + [col for col in prognosis_df.columns if col != "prognosis"]]
#sort the dataset alphabetically by prognosis
prognosis_df = prognosis_df.sort_values("prognosis")
#reset the index
# prognosis_df = prognosis_df.reset_index(drop=True)
prognosis_df.tail()

In [ ]:
# prognosis_df.set_index('prognosis', inplace=True)


In [ ]:
prognosis_df.head()

In [ ]:
# rename prognosis column to Prognosis
prognosis_df.rename(columns={'prognosis': 'Prognosis'}, inplace=True)
# reset the index
prognosis_df = prognosis_df.reset_index(drop=True)

In [ ]:
prognosis_df.head()

In [ ]:
prognosis_df.shape

In [ ]:
prognosis_df.describe()

In [ ]:
prognosis_df.info()

In [ ]:
prognosis_df.dtypes

In [ ]:
# read in prognosis csv file
final_train_df = pd.read_csv('Resources/FInal_Train_Data.csv')

# Display the result
final_train_df.head()

In [ ]:
final_train_df.shape

In [ ]:
#drop 'Unnamed: 0' column from the final_train_df dataframe
final_train_df.drop('Unnamed: 0', axis=1, inplace=True)

In [ ]:
# rename prognosis column to Prognosis
final_train_df.rename(columns={'prognosis': 'Prognosis'}, inplace=True)

# make prognosis column the first column in the dataset
final_train_df = final_train_df[["Prognosis"] + [col for col in final_train_df.columns if col != "Prognosis"]]

In [ ]:
final_train_df.sort_values('Prognosis', inplace=True)

In [ ]:
# reset the index
final_train_df = final_train_df.reset_index(drop=True)

In [ ]:
#display the first 5 rows of the dataframe
final_train_df.head()

In [ ]:
final_train_df.info()


In [ ]:
final_train_df.describe()

In [ ]:
final_train_df.dtypes

In [ ]:
# Example generator function handling all three datasets separately
def my_generator():
    for i in range(max(len(final_train_df), len(prognosis_df), len(disease_df_encoded))):
        yield (
            np.array(final_train_df.iloc[i % len(final_train_df)], dtype=np.float32),  # Features from final_train_df
            np.array(prognosis_df.iloc[i % len(prognosis_df)], dtype=np.int32),  # Labels from prognosis_df
            np.array(disease_df_encoded.iloc[i % len(disease_df_encoded)], dtype=np.float32)  # Encoded disease data
        )

# Define output_signature for all three datasets
output_signature = (
    tf.TensorSpec(shape=(final_train_df.shape[1],), dtype=tf.float32),  # Features vector
    tf.TensorSpec(shape=(), dtype=tf.int32),  # Prognosis label (assuming a single value)
    tf.TensorSpec(shape=(disease_df_encoded.shape[1],), dtype=tf.float32)  # Encoded disease vector
)

# Create a TensorFlow dataset
dataset = tf.data.Dataset.from_generator(my_generator, output_signature=output_signature)

# Iterate and print elements to verify
for element in dataset.take(5):  # Print first 5 elements for verification
    print(element)


In [ ]:
# Separate features and target labels
y = prognosis_df["Prognosis"].to_numpy()

X1 = prognosis_df.drop(columns=["Prognosis"]).to_numpy()
X2 = disease_df_encoded.drop(columns=["Prognosis"]).to_numpy()
X3 = final_train_df.drop(columns=["Prognosis"]).to_numpy()

In [ ]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

In [ ]:
# Train-test split for each input
X1_train, X1_test, y_train, y_test = train_test_split(X1, y_encoded, test_size=0.2, random_state=42)
X2_train, X2_test = train_test_split(X2, test_size=0.2, random_state=42)
X3_train, X3_test = train_test_split(X3, test_size=0.2, random_state=42)

In [ ]:
# Define input layers
input1 = Input(shape=(X1_train.shape[1],), name="prognosis_input")
input2 = Input(shape=(X2_train.shape[1],), name="disease_input")
input3 = Input(shape=(X3_train.shape[1],), name="final_train_input")

In [ ]:
# Define separate processing layers
x1 = Dense(64, activation="relu")(input1)
x2 = Dense(64, activation="relu")(input2)
x3 = Dense(64, activation="relu")(input3)

In [ ]:
# Concatenate processed inputs
merged = Concatenate()([x1, x2, x3])

In [ ]:
# Fully connected layers after merging
x = Dense(128, activation="relu")(merged)
x = Dense(64, activation="relu")(x)
output = Dense(len(np.unique(y)), activation="softmax")(x)  # Multi-class classification

In [ ]:
# Create model
model = tf.keras.Model(inputs=[input1, input2, input3], outputs=output)

In [ ]:
# Compile model
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

In [ ]:
# # Define a data generator for different batch sizes
def data_generator(X1, X2, X3, y, batch_size):
    while True:
        for i in range(0, min(len(X1), len(X2), len(X3)), batch_size):
            # Ensure batches are correctly sized and handled
            X1_batch = np.array(X1[i:i+batch_size], dtype=np.float32)
            X2_batch = np.array(X2[i:i+batch_size], dtype=np.float32) if i < len(X2) else np.zeros((batch_size, X2.shape[1]))
            X3_batch = np.array(X3[i:i+batch_size], dtype=np.float32) if i < len(X3) else np.zeros((batch_size, X3.shape[1]))
            y_batch = np.array(y[i:i+batch_size], dtype=np.int32)  # Ensure `y` is an integer array

            # Yield as tuple of tensors
            yield (X1_batch, X2_batch, X3_batch), y_batch

In [ ]:
# Set batch size
batch_size = 32

# Initialize generator
train_generator = data_generator(X1_train, X2_train, X3_train, y_train, batch_size)

# Convert to a TensorFlow dataset (optional)
train_dataset = tf.data.Dataset.from_generator(
    lambda: train_generator,
    output_signature=(
        (
            tf.TensorSpec(shape=(None, X1_train.shape[1]), dtype=tf.float32),
            tf.TensorSpec(shape=(None, X2_train.shape[1]), dtype=tf.float32),
            tf.TensorSpec(shape=(None, X3_train.shape[1]), dtype=tf.float32),
        ),
        tf.TensorSpec(shape=(None,), dtype=tf.int32),
    )
)

In [ ]:
# Train model using dataset
model.fit(train_dataset, epochs=20, steps_per_epoch=len(y_train) // batch_size)

In [ ]:
# # Define a data generator for different batch sizes
# def data_generator(X1, X2, X3, y, batch_size):
#     while True:
#         for i in range(0, min(len(X1), len(X2), len(X3)), batch_size):
#             X1_batch = X1[i:i+batch_size]
#             X2_batch = X2[i:i+batch_size] if i < len(X2) else np.zeros((batch_size, X2.shape[1]))
#             X3_batch = X3[i:i+batch_size] if i < len(X3) else np.zeros((batch_size, X3.shape[1]))
#             y_batch = y[i:i+batch_size]
#             yield [X1_batch, X2_batch, X3_batch], y_batch

# batch_size = 32
# train_generator = data_generator(X1_train, X2_train, X3_train, y_train, batch_size)

In [ ]:
# Train model using generator
# model.fit(train_generator, steps_per_epoch=len(y_train) // batch_size, epochs=20)